# Custom Tools in CrewAI [Step 1 -- Building your own tools]

> **MLCourse - Agentic AI - CrewAI Advanced Agents**

CrewAI agents gain power through tools. Built-in tools cover common tasks, but
real projects demand custom tools that understand YOUR data and domain. This
notebook shows two authoring patterns: the lightweight `@tool` decorator for
simple functions, and the `BaseTool` subclass for complex stateful tools.

## What you will learn

- `@tool` decorator: wrap any Python function into a CrewAI tool in one line
- `BaseTool` subclass: full control over `_run()`, input schemas, and error handling
- Error handling inside tools: graceful failures that do not crash the crew
- `force_tool_output_as_result`: force tool output to be the task result directly
- Tool call hooks: pre/post execution callbacks for logging and metrics
- `tool.run()` standalone: invoke a tool outside of any agent or crew

In [ ]:
# === SETUP CELL ===
import os
import re
from pathlib import Path

from dotenv import load_dotenv

# Walk up from cwd until we reach the track root folder "03_agentic_ai".
# This lets the notebook run from any subfolder while finding the shared .env.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

# Guard Jupyter-only magic so this file stays valid as plain Python too.
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("Setup complete. Track root resolved to:", TRACK)

## 1. The `@tool` decorator -- simplest path to a custom tool

CrewAI's `@tool` decorator from `crewai.tools` converts any function into a
tool that an agent can call. The function docstring becomes the tool description
the LLM reads when deciding which tool to use. Keep docstrings precise and
action-oriented -- agents pick tools based on these descriptions.

**Key rules:**
- Type hints on parameters define the input schema the LLM sees
- Return value MUST be a string (CrewAI serializes tool outputs as text)
- Raise `ToolExecutionError` for recoverable failures, or let exceptions propagate

In [ ]:
from crewai.tools import tool


# Word counter tool: counts words, characters, and sentences in a text block.
# The docstring is the ONLY thing the LLM sees when choosing this tool, so it
# must clearly describe what the tool does and what input it expects.
@tool("word_counter")
def word_counter(text: str) -> str:
    """Count the number of words, characters, and sentences in the given text.
    Input should be a string of text to analyze."""
    words = len(text.split())
    chars = len(text)
    # Split on sentence-ending punctuation to approximate sentence count.
    sentences = len(re.split(r'[.!?]+', text.strip()))
    # Strip trailing empty from the split.
    sentences = max(sentences, 1) if text.strip() else 0
    result = f"Words: {words}, Characters: {chars}, Sentences: {sentences}"
    return result


# Verify the tool works standalone -- no agent or crew needed.
sample = "CrewAI makes building AI agents straightforward. It handles orchestration."
print("Tool name:", word_counter.name)
print("Tool description:", word_counter.description)
print("Standalone call:", word_counter.run(sample))

## 2. A second `@tool` -- text summarizer

Simple tools can still be powerful when they encapsulate a focused operation.
This summarizer extracts the first N sentences as a naive extractive summary.
A production tool might call an LLM or use a dedicated summarization model,
but the wiring pattern is identical.

In [ ]:
@tool("text_summarizer")
def text_summarizer(text: str, num_sentences: int = 2) -> str:
    """Extract a summary from the given text by returning the first N sentences.
    Input is the full text and an optional sentence count (default 2)."""
    # Split into sentences on common sentence boundaries.
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    # Clamp to available sentences so we never index out of range.
    selected = sentences[:num_sentences]
    summary = " ".join(selected)
    return summary if summary else "No sentences found in input."


# Test with a longer passage to see extractive summarization in action.
long_text = (
    "CrewAI is a framework for orchestrating role-playing autonomous AI agents. "
    "These agents collaborate on complex tasks by dividing responsibilities. "
    "Each agent has a defined role, goal, and backstory that guide its behavior. "
    "The framework supports tool use, memory, and human-in-the-loop patterns. "
    "Crews can run sequentially or in parallel depending on the process type."
)
print("Full text length:", len(long_text), "chars")
print("Summary (2 sentences):", text_summarizer.run(long_text))
print("Summary (1 sentence):", text_summarizer.run(long_text, num_sentences=1))

## 3. A third `@tool` -- regex extractor

Pattern extraction is a common agent need: pull emails from a document, find
URLs in scraped content, or extract structured data from unstructured text.
This tool accepts a text and a regex pattern, returning all matches.

In [ ]:
@tool("regex_extractor")
def regex_extractor(text: str, pattern: str = r'\b\w+@\w+\.\w+\b') -> str:
    """Extract all matches of a regex pattern from the given text.
    Input is the text and an optional regex pattern (default finds email addresses).
    Returns a comma-separated list of matches or a 'no matches' message."""
    matches = re.findall(pattern, text)
    if not matches:
        return "No matches found for the given pattern."
    return ", ".join(matches)


# Test with default email pattern.
sample_text = (
    "Contact us at support@example.com or sales@company.org. "
    "For urgent issues, email help@service.net directly."
)
print("Emails found:", regex_extractor.run(sample_text))

# Test with a custom pattern for URLs.
url_text = (
    "Visit https://example.com for details. "
    "Also check http://docs.crewai.com and https://github.com/crewAI."
)
print("URLs found:", regex_extractor.run(url_text, pattern=r'https?://\S+'))

## 4. `BaseTool` subclass -- full control

When `@tool` is not enough -- you need custom validation, state, or complex
logic -- subclass `BaseTool` directly. Override `_run()` with your logic, and
define a Pydantic model for the args schema if you need typed, validated inputs.

**Advantages over `@tool`:**
- Custom input validation via Pydantic `args_schema`
- Access to `self` for maintaining state across calls
- Override `_run()` with full exception control
- Can implement `__init__` for configuration

In [ ]:
from crewai.tools import BaseTool
from pydantic import BaseModel, Field


# Define the input schema as a Pydantic model -- CrewAI inspects this to build
# the tool description and validate LLM-provided arguments before _run() fires.
class RegexExtractInput(BaseModel):
    """Input schema for the regex extraction tool."""
    text: str = Field(description="The text to search through.")
    pattern: str = Field(
        default=r'\b\d{3}-\d{4}\b',
        description="The regex pattern to search for (default: phone numbers like 555-1234)."
    )


class AdvancedRegexExtractor(BaseTool):
    """A stateful regex extractor that tracks match counts across invocations.

    This demonstrates the BaseTool subclass pattern: define args_schema for
    validated inputs, override _run() for the actual logic, and use self.*
    for any state that persists across calls within the same agent session.
    """
    name: str = "advanced_regex_extractor"
    description: str = (
        "Extract regex matches from text with full pattern support. "
        "Returns matches and running statistics of total extractions performed."
    )
    # Link the Pydantic schema so CrewAI can validate inputs.
    args_schema: type[BaseModel] = RegexExtractInput

    # Internal counter: persists across calls within the same tool instance.
    _total_matches: int = 0

    def _run(self, text: str, pattern: str = r'\b\d{3}-\d{4}\b') -> str:
        """Execute regex extraction and update internal statistics."""
        try:
            matches = re.findall(pattern, text)
            self._total_matches += len(matches)
            if not matches:
                return f"No matches found. Total matches across all calls: {self._total_matches}"
            match_list = ", ".join(matches)
            return (
                f"Matches: {match_list}\n"
                f"This call: {len(matches)} | Running total: {self._total_matches}"
            )
        except re.error as e:
            return f"Invalid regex pattern: {e}. Please provide a valid Python regex."


# Instantiate and test the BaseTool subclass.
advanced_tool = AdvancedRegexExtractor()

# First call -- extracts phone numbers from sample text.
sample_phones = (
    "Office: 555-1234, Mobile: 555-5678, Fax: 555-9012. "
    "Emergency: call 911 (not a match)."
)
print("--- Call 1 ---")
print(advanced_tool.run(sample_phones))

# Second call -- the running total persists from the first call.
print("\n--- Call 2 ---")
print(advanced_tool.run("Back office: 555-3456"))

## 5. Error handling inside tools

Tools WILL fail: invalid inputs, network timeouts, bad regex, missing files.
How you handle errors determines whether the agent retries, pivots, or crashes.
CrewAI catches `ToolExecutionError` and feeds the error message back to the LLM,
which can then reason about what went wrong and try a different approach.

In [ ]:
from crewai.tools import ToolExecutionError


@tool("safe_divider")
def safe_divider(numerator: str, denominator: str) -> str:
    """Divide two numbers provided as strings. Returns the result or an error message.
    Input is a numerator and denominator, both as string representations of numbers."""
    try:
        a = float(numerator)
        b = float(denominator)
    except ValueError:
        # Return a human-readable error -- the LLM will see this and adjust.
        return "Error: both inputs must be valid numbers."
    if b == 0:
        # Division by zero: explicit message so the agent knows NOT to retry the same values.
        return "Error: division by zero is undefined. Choose a non-zero denominator."
    return f"{a} / {b} = {a / b}"


# Test normal operation.
print("Normal:", safe_divider.run("10", "3"))

# Test non-numeric input -- the tool returns a helpful message, not a traceback.
print("Bad input:", safe_divider.run("abc", "5"))

# Test division by zero -- graceful handling instead of a crash.
print("Div by zero:", safe_divider.run("7", "0"))

## 6. Tool call hooks -- pre/post execution callbacks

Hooks let you inject cross-cutting concerns without modifying the tool logic:
logging, metrics, rate limiting, caching, or audit trails. CrewAI fires hooks
before and after every `_run()` call. Both hooks receive the same arguments
that the tool itself receives, plus the tool instance.

In [ ]:
import time


# A simple hook list that records timestamps and durations.
execution_log = []


def pre_run_hook(tool_instance, **kwargs):
    """Called BEFORE the tool executes. Log the start time and input summary."""
    entry = {
        "tool": tool_instance.name,
        "start": time.time(),
        "input_preview": str(kwargs)[:80],
    }
    execution_log.append(entry)
    print(f"[hook] pre_run: {tool_instance.name} called with {list(kwargs.keys())}")


def post_run_hook(tool_instance, output, **kwargs):
    """Called AFTER the tool executes. Compute duration and log the result."""
    if execution_log:
        last = execution_log[-1]
        last["duration_ms"] = round((time.time() - last["start"]) * 1000, 2)
        last["output_preview"] = str(output)[:80]
    print(f"[hook] post_run: {tool_instance.name} completed in {execution_log[-1]['duration_ms']}ms")


# Attach hooks to the word_counter tool instance.
word_counter.pre_run = pre_run_hook
word_counter.post_run = post_run_hook

# Execute with hooks active -- you should see hook output around the tool result.
result = word_counter.run("This text has hooks attached to the tool invocation.")
print("Result:", result)
print("\nExecution log:", execution_log)

## 7. Standalone `tool.run()` -- no agent needed

Tools are standalone callable objects. You can `.run()` them directly without
an agent or crew, which is invaluable for:
- Unit testing tool logic in isolation
- Prototyping before wiring into a crew
- Calling tools from non-CrewAI code paths

The `.run()` method accepts keyword arguments matching the function signature
(for `@tool`) or the Pydantic schema fields (for `BaseTool` subclasses).

In [ ]:
# Demonstrate standalone invocation of all three custom tools.
print("=== word_counter.run() ===")
print(word_counter.run("Testing standalone tool execution."))

print("\n=== text_summarizer.run() ===")
demo_text = (
    "Machine learning is a subset of artificial intelligence. "
    "It focuses on building systems that learn from data. "
    "Deep learning is a subset of machine learning using neural networks. "
    "Transformers revolutionized natural language processing in 2017."
)
print(text_summarizer.run(demo_text, num_sentences=3))

print("\n=== regex_extractor.run() ===")
print(regex_extractor.run("Errors at 2024-01-15 and 2024-03-22", pattern=r'\d{4}-\d{2}-\d{2}'))

print("\n=== advanced_regex_extractor.run() ===")
print(advanced_regex_extractor.run("Call 555-0001 or 555-0002", pattern=r'\b\d{3}-\d{4}\b'))

## 8. Wiring custom tools into a CrewAI agent

Now let us assemble a minimal crew that uses the custom tools. The agent
receives the tools list and the LLM decides which tool to call based on the
task description and tool docstrings. This is where your tool descriptions
pay off -- clear descriptions mean the LLM picks the right tool more often.

In [ ]:
from crewai import Agent, Task, Crew, Process

# Create a research analyst agent with our custom tools attached.
# The agent uses ChatOllama (local, free) so no API key is required.
research_agent = Agent(
    role="Research Analyst",
    goal="Analyze and extract information from text documents accurately.",
    backstory=(
        "You are a meticulous analyst who uses precise tools to measure text "
        "properties, extract patterns, and summarize key findings."
    ),
    tools=[word_counter, text_summarizer, regex_extractor],
    # Use ChatOllama for a fully local, free setup -- no API keys needed.
    llm="ollama/llama3.1:8b",
    verbose=False,
    allow_delegation=False,
)

# Define a task that exercises multiple tools.
analysis_task = Task(
    description=(
        "Analyze the following text and provide a structured report:\n"
        "Text: 'Artificial intelligence is transforming healthcare. AI systems can "
        "detect diseases earlier than human doctors in some cases. The global AI "
        "healthcare market is expected to reach 187 billion dollars by 2030. "
        "Key players include Google Health, Microsoft, and IBM Watson. Contact "
        "info@aihealth.example.com for partnership inquiries.'\n\n"
        "Report should include: word count, a 1-sentence summary, and any "
        "email addresses found in the text."
    ),
    expected_output=(
        "A structured report with word count statistics, a summary, "
        "and extracted email addresses."
    ),
    agent=research_agent,
)

# Build and run the crew.
crew = Crew(
    agents=[research_agent],
    tasks=[analysis_task],
    process=Process.sequential,
    verbose=False,
)

try:
    result = crew.kickoff()
    print("=== Crew Result ===")
    print(result)
except Exception as e:
    print(f"[demo skipped] CrewAI crew execution failed: {e}")

## 9. `force_tool_output_as_result`

By default, CrewAI sends the tool output to the LLM for further processing.
Sometimes you want the raw tool output to BE the final task result, bypassing
the LLM summarization step. Setting `force_tool_output_as_result=True` on a
task tells CrewAI to use the tool's string output directly as the task output.

In [ ]:
# Create a task that uses the word counter tool and forces its output as the result.
# This bypasses LLM post-processing -- the exact tool output becomes the task result.
counting_agent = Agent(
    role="Text Counter",
    goal="Count words and sentences precisely using the word_counter tool.",
    backstory="You are a precise counter who always returns exact statistics.",
    tools=[word_counter],
    llm="ollama/llama3.1:8b",
    verbose=False,
    allow_delegation=False,
)

counting_task = Task(
    description=(
        "Count the words in this text using the word_counter tool: "
        "'The quick brown fox jumps over the lazy dog. Pack my box with "
        "five dozen liquor jugs.'"
    ),
    expected_output="Exact word count, character count, and sentence count.",
    agent=counting_agent,
    # Force the tool output to be the result -- no LLM rewriting.
    force_tool_output_as_result=True,
)

force_crew = Crew(
    agents=[counting_agent],
    tasks=[counting_task],
    process=Process.sequential,
    verbose=False,
)

try:
    force_result = force_crew.kickoff()
    print("=== force_tool_output_as_result ===")
    print(force_result)
except Exception as e:
    print(f"[demo skipped] Crew execution failed: {e}")

## Summary and key takeaways

- `@tool` decorator is the fastest path: write a function, add type hints and
  a docstring, and the tool is ready for any CrewAI agent.
- `BaseTool` subclass gives full control: custom validation via `args_schema`,
  persistent state via `self`, and structured error handling in `_run()`.
- Error handling inside tools determines crew resilience: return descriptive
  error strings so the LLM can reason about failures and retry intelligently.
- `force_tool_output_as_result` bypasses LLM post-processing when you need
  exact tool output as the task result.
- Tool call hooks enable cross-cutting concerns (logging, metrics) without
  touching tool logic.
- `tool.run()` works standalone -- no agent or crew required -- making
  unit testing and prototyping straightforward.

**Next up:** notebook 02 covers Knowledge Sources for grounding agent reasoning
in external documents.